In [0]:
%run /Workspace/Data_Ingestion_Atlikon/utilities

In [0]:
print("bronze_schema: ", bronze_schema)
print("silver_schema: ", silver_schema)
print("gold_schema: ", gold_schema)

bronze_schema:  bronze
silver_schema:  silver
gold_schema:  gold


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
# Creates a text input widget named "catalog" with default value "fmcg" and label "Catalog"
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://childcompanydata-myfirstproject/{data_source}/*.csv'
print(base_path)

s3://childcompanydata-myfirstproject/customers/*.csv


Load data in Bronze layer

In [0]:
df_customers = (spark.read.format("csv") \
                  .option("header", "true")\
                  .option("inferSchema", "true")\
                  .load(base_path)\
                  .withColumn("read_timestamp",F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss.SSS"))\
                  .select("*" , "_metadata.file_name", "_metadata.file_size", "_metadata.file_path")
                  )
                  
display(df_customers.limit(10))

customer_id,customer_name,city,read_timestamp,file_name,file_size,file_path
789201,FitFuel Market,Bengaluru,2025-12-12 18:34:13.733,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv
789202,FitFuel Market,Hyderabad,2025-12-12 18:34:13.733,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv
789203,FitFuel Market,New Delhi,2025-12-12 18:34:13.733,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv
789301,Athlete's Choice Store,Bengaluru,2025-12-12 18:34:13.733,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv
789303,Athlete's Choice Store,New Delhi,2025-12-12 18:34:13.733,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv
789101,Endurance Foods,Bengalore,2025-12-12 18:34:13.733,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv
789102,Endurance Foods,Hyderabad,2025-12-12 18:34:13.733,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv
789103,Endurance Foods,New Delhi,2025-12-12 18:34:13.733,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv
789121,HydroBoost Nutrition,Hyderabad,2025-12-12 18:34:13.733,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv
789122,HydroBoost Nutrition,New Delhi,2025-12-12 18:34:13.733,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv


In [0]:
df_customers.write\
            .format("delta")\
            .option("delta.enableChangeDataFeed", "true")\
            .mode("overwrite")\
            .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

## Silver Layer

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

+-----------+--------------------+---------+--------------------+-------------+---------+--------------------+
|customer_id|       customer_name|     city|      read_timestamp|    file_name|file_size|           file_path|
+-----------+--------------------+---------+--------------------+-------------+---------+--------------------+
|     789201|      FitFuel Market|Bengaluru|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789202|      FitFuel Market|Hyderabad|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789203|      FitFuel Market|New Delhi|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789301|Athlete's Choice ...|Bengaluru|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789303|Athlete's Choice ...|New Delhi|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789101|     Endurance Foods|Bengalore|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|

In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count") > 1)
df_duplicates.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|     789321|    2|
|     789503|    2|
|     789522|    2|
|     789603|    2|
+-----------+-----+



In [0]:
df_silver = df_bronze.dropDuplicates(["customer_id"])
df_silver.show(10)

+-----------+--------------------+---------+--------------------+-------------+---------+--------------------+
|customer_id|       customer_name|     city|      read_timestamp|    file_name|file_size|           file_path|
+-----------+--------------------+---------+--------------------+-------------+---------+--------------------+
|     789201|      FitFuel Market|Bengaluru|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789202|      FitFuel Market|Hyderabad|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789203|      FitFuel Market|New Delhi|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789301|Athlete's Choice ...|Bengaluru|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789303|Athlete's Choice ...|New Delhi|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789101|     Endurance Foods|Bengalore|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|

In [0]:
display(df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name"))))

customer_id,customer_name,city,read_timestamp,file_name,file_size,file_path
789121,HydroBoost Nutrition,Hyderabad,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv
789401,SprintX nutrition,Bengaluru,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv
789420,ZenAthlete foods,null,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv
789421,ZenAthlete Foods,Hyderbad,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv
789521,PrimeFuel Nutrition,null,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv
789702,StaminaX Store,Hyderabad,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv


In [0]:
df_silver = df_silver.withColumn("customer_name", F.trim(F.col("customer_name")))
display(df_silver.count())

35

In [0]:
df_silver.select("city").distinct().show()

+----------+
|      city|
+----------+
| Bengaluru|
| Hyderabad|
| New Delhi|
| Bengalore|
|Hyderabadd|
|      NULL|
|  Hyderbad|
| NewDelhee|
|  NewDelhi|
|Bengaluruu|
|  NewDheli|
+----------+



In [0]:
%python

city_mapping= {
    "Bengalore": "Bangalore",
    "Bengaluruu": "Bangalore",
    "Hyderabadd": "Hyderabad",
    "Hyderbad": "Hyderabad",
    "NewDelhi": "New Delhi",
    "NewDheli": "New Delhi"
}

allowed_city = ["Bangalore", "Hyderabad", "New Delhi"]

df_silver = (df_silver
             .replace(city_mapping, subset=["city"])
             .withColumn(
                 "city",
                 F.when(F.col("city").isNull(),None)
                 .when(F.col("city").isin(allowed_city), F.col("city"))
                 .otherwise(None)
                )
            )

# df_silver.select("city").distinct().show()

In [0]:
df_silver.select("city").distinct().show()

+---------+
|     city|
+---------+
|     NULL|
|Hyderabad|
|New Delhi|
|Bangalore|
+---------+



In [0]:
# df_silver.select("customer_name").distinct().show()

df_silver = df_silver.withColumn("customer_name", 
                                 F.when(F.col("customer_name").isNull(), None)
                                 .otherwise(F.initcap("customer_name"))
                                 )

df_silver.select("customer_name").distinct().show()

+--------------------+
|       customer_name|
+--------------------+
|      Fitfuel Market|
|Athlete's Choice ...|
|     Endurance Foods|
|Hydroboost Nutrition|
|Macrobite Superfoods|
|      Powersnack Hub|
|   Sprintx Nutrition|
|    Zenathlete Foods|
|Peak Performance ...|
| Primefuel Nutrition|
|       Recovery Lane|
|      Staminax Store|
|Eliteathlete Nutr...|
|      Gameplan Foods|
|   Champion's Choice|
+--------------------+

+--------------------+
|       customer_name|
+--------------------+
|      Fitfuel Market|
|Athlete's Choice ...|
|     Endurance Foods|
|Hydroboost Nutrition|
|Macrobite Superfoods|
|      Powersnack Hub|
|   Sprintx Nutrition|
|    Zenathlete Foods|
|Peak Performance ...|
| Primefuel Nutrition|
|       Recovery Lane|
|      Staminax Store|
|Eliteathlete Nutr...|
|      Gameplan Foods|
|   Champion's Choice|
+--------------------+



In [0]:
df_silver.filter(F.col("city").isNull()).show()

df_null_customer_cities = df_silver.filter(F.col("city").isNull()).select("customer_name").distinct()

# Assuming df has one column named "col_name"
null_customer_cities = [row["customer_name"] for row in df_null_customer_cities.select("customer_name").collect()]

+-----------+--------------------+----+--------------------+-------------+---------+--------------------+
|customer_id|       customer_name|city|      read_timestamp|    file_name|file_size|           file_path|
+-----------+--------------------+----+--------------------+-------------+---------+--------------------+
|     789201|      Fitfuel Market|NULL|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789301|Athlete's Choice ...|NULL|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789220|Macrobite Superfoods|NULL|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789320|      Powersnack Hub|NULL|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789401|   Sprintx Nutrition|NULL|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789403|   Sprintx Nutrition|NULL|2025-12-12 18:34:...|customers.csv|     1404|s3://childcompany...|
|     789420|    Zenathlete Foods|NULL|2025-12

In [0]:
display(null_customer_cities)

_1
Fitfuel Market
Athlete's Choice Store
Macrobite Superfoods
Powersnack Hub
Sprintx Nutrition
Zenathlete Foods
Peak Performance Store
Primefuel Nutrition
Recovery Lane
Gameplan Foods


In [0]:
df_silver.filter(F.col("customer_name").isin(null_customer_cities)).select(["customer_id","customer_name","city"]).show(35)

+-----------+--------------------+---------+
|customer_id|       customer_name|     city|
+-----------+--------------------+---------+
|     789201|      Fitfuel Market|     NULL|
|     789202|      Fitfuel Market|Hyderabad|
|     789203|      Fitfuel Market|New Delhi|
|     789301|Athlete's Choice ...|     NULL|
|     789303|Athlete's Choice ...|New Delhi|
|     789220|Macrobite Superfoods|     NULL|
|     789221|Macrobite Superfoods|Hyderabad|
|     789320|      Powersnack Hub|     NULL|
|     789321|      Powersnack Hub|Hyderabad|
|     789401|   Sprintx Nutrition|     NULL|
|     789402|   Sprintx Nutrition|Hyderabad|
|     789403|   Sprintx Nutrition|     NULL|
|     789420|    Zenathlete Foods|     NULL|
|     789421|    Zenathlete Foods|Hyderabad|
|     789422|    Zenathlete Foods|     NULL|
|     789501|Peak Performance ...|     NULL|
|     789503|Peak Performance ...|New Delhi|
|     789520| Primefuel Nutrition|Bangalore|
|     789521| Primefuel Nutrition|     NULL|
|     7895

In [0]:
fix_null_city = {
    789201: "Bangalore",
    789301: "Bangalore",
    789220: "Bangalore",
    789320: "Bangalore",
    789403: "Bangalore",
    789420: "Bangalore",
    789501: "Bangalore",
    789601: "Bangalore",
    789720: "Bangalore",

    789401: "New Delhi",
    789422: "New Delhi",
    789603: "New Delhi",

    789521: "Hyderabad"
}

df_customer_city_fix = spark.createDataFrame(
    [(k,v) for k,v in fix_null_city.items()],
    ["customer_id", "fixed_city"]
)

display(df_customer_city_fix)

customer_id,fixed_city
789201,Bangalore
789301,Bangalore
789220,Bangalore
789320,Bangalore
789403,Bangalore
789420,Bangalore
789501,Bangalore
789601,Bangalore
789720,Bangalore
789401,New Delhi


In [0]:
df_silver=df_silver.join(df_customer_city_fix, on="customer_id", how="left")\
    .withColumn("city", F.coalesce(F.col("fixed_city"), F.col("city")))\
    .drop("fixed_city")

df_silver.filter(F.col("city").isNull()).display()

customer_id,customer_name,city,read_timestamp,file_name,file_size,file_path


In [0]:
df_silver= df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))

df_silver.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: string (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)
 |-- file_path: string (nullable = true)



In [0]:
df_silver= (df_silver
            .withColumn("customer", 
                        F.concat_ws("-", F.col("customer_name"), F.coalesce(F.col("city"), F.lit('Unknown'))
                                    )
                        )
            .withColumn("market", F.lit("India"))
            .withColumn("platform", F.lit("Sports Bar"))
            .withColumn("channel", F.lit("Acquisition"))
            ) 

display(df_silver.limit(10))


customer_id,customer_name,city,read_timestamp,file_name,file_size,file_path,customer,market,platform,channel
789503,Peak Performance Store,New Delhi,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv,Peak Performance Store-New Delhi,India,Sports Bar,Acquisition
789420,Zenathlete Foods,Bangalore,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv,Zenathlete Foods-Bangalore,India,Sports Bar,Acquisition
789703,Staminax Store,New Delhi,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv,Staminax Store-New Delhi,India,Sports Bar,Acquisition
789621,Eliteathlete Nutrition,Hyderabad,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv,Eliteathlete Nutrition-Hyderabad,India,Sports Bar,Acquisition
789101,Endurance Foods,Bangalore,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv,Endurance Foods-Bangalore,India,Sports Bar,Acquisition
789220,Macrobite Superfoods,Bangalore,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv,Macrobite Superfoods-Bangalore,India,Sports Bar,Acquisition
789720,Gameplan Foods,Bangalore,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv,Gameplan Foods-Bangalore,India,Sports Bar,Acquisition
789601,Recovery Lane,Bangalore,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv,Recovery Lane-Bangalore,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition,New Delhi,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789402,Sprintx Nutrition,Hyderabad,2025-12-12 18:34:32.596,customers.csv,1404,s3://childcompanydata-myfirstproject/customers/customers.csv,Sprintx Nutrition-Hyderabad,India,Sports Bar,Acquisition


In [0]:
df_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "True") \
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

##  Gold Layer

In [0]:
df_silver = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")

df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold = df_gold.withColumnRenamed("customer_id", "customer_code")

df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "True")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")